# Saree Virtual Try-On — VITON-HD Style Training
**Steps:**
1. Install dependencies
2. Download & clean datasets from Kaggle
3. Build Cloth Warping Network (GMM) + Try-On Module (TOM)
4. Train on saree images
5. Save & export model weights

**GPU:** Enable T4 or P100 in Kaggle Settings → Accelerator

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q kaggle opencv-python-headless scikit-image albumentations
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q kornia tqdm matplotlib Pillow
import subprocess, sys
print('Python:', sys.version)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Step 2: Dataset Download & Setup

In [ ]:
import os, json

# ── Option A: Use Kaggle API (paste your kaggle.json content below) ────────────
KAGGLE_JSON = {
    'username': 'YOUR_KAGGLE_USERNAME',   # <-- Replace with your username
    'key':      'YOUR_KAGGLE_API_KEY'      # <-- Replace with your API key
}
# Get from: https://www.kaggle.com/settings → API → Create New Token

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(KAGGLE_JSON, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured')

In [ ]:
# Download saree datasets from Kaggle
# We'll use multiple complementary datasets:

os.makedirs('/kaggle/working/datasets/raw', exist_ok=True)

datasets = [
    # Primary: Saree fashion dataset with person+garment pairs
    ('manishkumar2311/indian-traditional-women-saree-fashion', 'saree_fashion'),
    # Secondary: Women fashion for person images
    ('paramaggarwal/fashion-product-images-small', 'fashion_persons'),
    # Tertiary: VITON benchmark for training structure reference
    ('naveengowda18/viton-dataset-for-virtual-try-on', 'viton_ref'),
]

for dataset_slug, folder_name in datasets:
    dest = f'/kaggle/working/datasets/raw/{folder_name}'
    if not os.path.exists(dest):
        print(f'Downloading {dataset_slug}...')
        ret = os.system(f'kaggle datasets download -d {dataset_slug} -p {dest} --unzip -q')
        if ret == 0:
            files = sum(len(f) for _, _, f in os.walk(dest))
            print(f'  ✓ Downloaded: {files} files')
        else:
            print(f'  ✗ Failed — dataset may not exist, trying alternative...')
    else:
        print(f'  Already exists: {dest}')

In [ ]:
# Alternative: If Kaggle datasets above fail, scrape/use these public sources
import urllib.request

FALLBACK_DATASETS = {
    # DeepFashion2 subset (person images, good body pose variety)
    'deepfashion_url': None,  # Too large for download here
    
    # Use what's already in /kaggle/input if run as Kaggle notebook
    'kaggle_input_path': '/kaggle/input',
}

# Check what's available in /kaggle/input (if datasets added via UI)
input_path = '/kaggle/input'
if os.path.exists(input_path):
    datasets_in_input = os.listdir(input_path)
    print(f'Datasets in /kaggle/input: {datasets_in_input}')
else:
    print('No /kaggle/input — using downloaded datasets')

# Count all available images
total_imgs = 0
for root, dirs, files in os.walk('/kaggle/working/datasets/raw'):
    img_files = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
    total_imgs += len(img_files)
print(f'Total images found: {total_imgs}')

## Step 3: Dataset Cleaning & Preprocessing

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import shutil
from tqdm import tqdm

TARGET_SIZE = (256, 192)  # H x W — standard VITON size
MIN_SIZE    = 128         # Minimum acceptable dimension

CLEAN_DIR   = Path('/kaggle/working/datasets/clean')
PERSON_DIR  = CLEAN_DIR / 'person'
GARMENT_DIR = CLEAN_DIR / 'garment'
PAIR_DIR    = CLEAN_DIR / 'pairs'

for d in [PERSON_DIR, GARMENT_DIR, PAIR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Clean dataset directories created')


def is_valid_image(path: str) -> bool:
    """Check if image is valid: readable, minimum size, not corrupted."""
    try:
        img = cv2.imread(path)
        if img is None:
            return False
        h, w = img.shape[:2]
        if h < MIN_SIZE or w < MIN_SIZE:
            return False
        # Check not blank (near-zero variance)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        if gray.std() < 5.0:
            return False
        return True
    except Exception:
        return False


def preprocess_image(path: str, out_path: str, size=(192, 256)) -> bool:
    """Resize with letterboxing, normalize, save."""
    try:
        img = Image.open(path).convert('RGB')
        w, h = img.size
        target_w, target_h = size
        
        # Letterbox resize preserving aspect ratio
        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)
        img_resized = img.resize((new_w, new_h), Image.LANCZOS)
        
        # Paste onto white canvas
        canvas = Image.new('RGB', (target_w, target_h), (255, 255, 255))
        offset = ((target_w - new_w) // 2, (target_h - new_h) // 2)
        canvas.paste(img_resized, offset)
        canvas.save(out_path, quality=95)
        return True
    except Exception as e:
        return False

print('Functions defined. Ready for cleaning.')

In [ ]:
# Collect and clean ALL images from downloaded datasets
RAW_DIRS = list(Path('/kaggle/working/datasets/raw').rglob('*.jpg')) + \
           list(Path('/kaggle/working/datasets/raw').rglob('*.jpeg')) + \
           list(Path('/kaggle/working/datasets/raw').rglob('*.png'))

# Also check /kaggle/input
if os.path.exists('/kaggle/input'):
    RAW_DIRS += list(Path('/kaggle/input').rglob('*.jpg')) + \
                list(Path('/kaggle/input').rglob('*.jpeg')) + \
                list(Path('/kaggle/input').rglob('*.png'))

print(f'Total raw images: {len(RAW_DIRS)}')

valid_count  = 0
invalid_count = 0
person_imgs  = []
garment_imgs = []

for img_path in tqdm(RAW_DIRS, desc='Validating images'):
    path_str = str(img_path)
    if not is_valid_image(path_str):
        invalid_count += 1
        continue
    valid_count += 1
    
    # Heuristic classification: garment images are usually square/wider
    # person images are usually taller (portrait)
    try:
        img = cv2.imread(path_str)
        h, w = img.shape[:2]
        aspect = h / w
        
        # Check for skin tones to detect person images
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        skin_mask = cv2.inRange(hsv,
            np.array([0, 20, 70], dtype=np.uint8),
            np.array([25, 255, 255], dtype=np.uint8))
        skin_frac = skin_mask.sum() / (h * w * 255)
        
        if skin_frac > 0.03 and aspect > 1.2:   # Tall image with skin = person
            person_imgs.append(path_str)
        elif aspect > 0.7 and skin_frac < 0.05: # Square/wide, no skin = garment
            garment_imgs.append(path_str)
    except Exception:
        pass

print(f'Valid: {valid_count}  |  Invalid/skipped: {invalid_count}')
print(f'Person images: {len(person_imgs)}')
print(f'Garment images: {len(garment_imgs)}')

In [ ]:
# Process and save cleaned person images
print('Processing person images...')
person_saved = []
for i, src in enumerate(tqdm(person_imgs[:5000], desc='Person images')):
    dst = str(PERSON_DIR / f'person_{i:05d}.jpg')
    if preprocess_image(src, dst, size=(192, 256)):
        person_saved.append(dst)

print(f'Saved {len(person_saved)} person images')

# Process and save cleaned garment images
print('Processing garment images...')
garment_saved = []
for i, src in enumerate(tqdm(garment_imgs[:5000], desc='Garment images')):
    dst = str(GARMENT_DIR / f'garment_{i:05d}.jpg')
    if preprocess_image(src, dst, size=(192, 256)):
        garment_saved.append(dst)

print(f'Saved {len(garment_saved)} garment images')

# Create pairs (person, garment) — we'll use the minimum count
N_PAIRS = min(len(person_saved), len(garment_saved))
print(f'Creating {N_PAIRS} training pairs...')

pairs = []
for i in range(N_PAIRS):
    pairs.append({'person': person_saved[i], 'garment': garment_saved[i]})

# Save pairs index
import json
with open('/kaggle/working/datasets/clean/pairs.json', 'w') as f:
    json.dump(pairs, f, indent=2)

print(f'Dataset ready: {N_PAIRS} pairs')

In [ ]:
# Visualise sample cleaned images
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Cleaned Dataset Samples', fontsize=14)

for i in range(6):
    if i < len(person_saved):
        img = Image.open(person_saved[i])
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Person {i}', fontsize=8)
        axes[0, i].axis('off')
    if i < len(garment_saved):
        img = Image.open(garment_saved[i])
        axes[1, i].imshow(img)
        axes[1, i].set_title(f'Garment {i}', fontsize=8)
        axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/dataset_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('Sample visualisation saved.')

## Step 4: Pose Extraction for All Person Images

In [ ]:
# Extract pose keypoints for all person images using MediaPipe
!pip install -q mediapipe

import mediapipe as mp
import urllib.request

# Download pose landmarker model
MODEL_PATH = '/kaggle/working/pose_landmarker_full.task'
if not os.path.exists(MODEL_PATH):
    print('Downloading pose landmarker model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
        'pose_landmarker_full/float16/latest/pose_landmarker_full.task',
        MODEL_PATH
    )
    print('Downloaded.')

from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision.core.vision_task_running_mode import VisionTaskRunningMode

POSE_DIR = CLEAN_DIR / 'pose'
POSE_DIR.mkdir(exist_ok=True)

base_opts = mp_tasks.BaseOptions(model_asset_path=MODEL_PATH)
options   = mp_vision.PoseLandmarkerOptions(
    base_options=base_opts,
    running_mode=VisionTaskRunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.3,
)

LANDMARK_NAMES = [
    'nose','left_eye_inner','left_eye','left_eye_outer',
    'right_eye_inner','right_eye','right_eye_outer',
    'left_ear','right_ear','mouth_left','mouth_right',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_pinky','right_pinky',
    'left_index','right_index','left_thumb','right_thumb',
    'left_hip','right_hip','left_knee','right_knee',
    'left_ankle','right_ankle','left_heel','right_heel',
    'left_foot_index','right_foot_index'
]

pose_index = {}  # person_path -> pose_json_path
no_pose_count = 0

with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
    for person_path in tqdm(person_saved, desc='Extracting poses'):
        bgr = cv2.imread(person_path)
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result  = landmarker.detect(mp_img)
        
        if not result.pose_landmarks:
            no_pose_count += 1
            continue
        
        h, w = bgr.shape[:2]
        kp = {}
        for idx, lm in enumerate(result.pose_landmarks[0]):
            name = LANDMARK_NAMES[idx] if idx < len(LANDMARK_NAMES) else f'lm_{idx}'
            kp[name] = {'x': round(lm.x * w, 2), 'y': round(lm.y * h, 2),
                        'visibility': round(getattr(lm, 'visibility', 1.0), 3)}
        
        pose_fname = Path(person_path).stem + '_pose.json'
        pose_path  = str(POSE_DIR / pose_fname)
        with open(pose_path, 'w') as f:
            json.dump({'keypoints': kp, 'image_size': {'w': w, 'h': h}}, f)
        pose_index[person_path] = pose_path

print(f'Poses extracted: {len(pose_index)}')
print(f'No pose detected: {no_pose_count}')

# Save pose index
with open('/kaggle/working/datasets/clean/pose_index.json', 'w') as f:
    json.dump(pose_index, f)
print('Pose index saved.')

## Step 5: Build VITON-HD Style Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import kornia

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


# ── Geometric Matching Module (GMM) — Learns cloth warp ──────────────────────
class FeatureExtractor(nn.Module):
    """Shared feature extractor for person + garment images."""
    def __init__(self, in_channels=3, base_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, base_ch,    4, 2, 1), nn.BatchNorm2d(base_ch),    nn.LeakyReLU(0.2),
            nn.Conv2d(base_ch,     base_ch*2,  4, 2, 1), nn.BatchNorm2d(base_ch*2),  nn.LeakyReLU(0.2),
            nn.Conv2d(base_ch*2,   base_ch*4,  4, 2, 1), nn.BatchNorm2d(base_ch*4),  nn.LeakyReLU(0.2),
            nn.Conv2d(base_ch*4,   base_ch*8,  4, 2, 1), nn.BatchNorm2d(base_ch*8),  nn.LeakyReLU(0.2),
            nn.Conv2d(base_ch*8,   base_ch*8,  4, 2, 1), nn.BatchNorm2d(base_ch*8),  nn.LeakyReLU(0.2),
        )
    def forward(self, x):
        return self.net(x)


class CorrelationLayer(nn.Module):
    """Computes feature correlation between person and garment features."""
    def __init__(self):
        super().__init__()

    def forward(self, f1, f2):
        b, c, h, w = f1.shape
        f1 = F.normalize(f1.view(b, c, -1), dim=1)   # B x C x HW
        f2 = F.normalize(f2.view(b, c, -1), dim=1)   # B x C x HW
        corr = torch.bmm(f1.permute(0, 2, 1), f2)    # B x HW x HW
        corr = F.relu(corr)
        return corr.view(b, h * w, h, w)


class ThetaRegressor(nn.Module):
    """Predicts TPS (Thin Plate Spline) control point offsets for cloth warping."""
    def __init__(self, in_channels, grid_size=5):
        super().__init__()
        self.grid_size = grid_size
        n_pts = grid_size * grid_size * 2   # x,y for each control point
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, n_pts), nn.Tanh(),   # [-1, 1] normalised offsets
        )
    def forward(self, x):
        theta = self.regressor(x)
        return theta.view(-1, self.grid_size * self.grid_size, 2)


class GMM(nn.Module):
    """Geometric Matching Module: warps garment to fit body pose."""
    def __init__(self, grid_size=5, img_h=256, img_w=192):
        super().__init__()
        self.grid_size = grid_size
        self.img_h     = img_h
        self.img_w     = img_w

        self.person_extractor  = FeatureExtractor(in_channels=3 + 18)  # RGB + 18-ch pose heatmap
        self.garment_extractor = FeatureExtractor(in_channels=3)
        self.correlation       = CorrelationLayer()

        feat_h = img_h // 32
        feat_w = img_w // 32
        corr_channels = feat_h * feat_w

        self.theta_reg = ThetaRegressor(corr_channels * feat_h * feat_w, grid_size)

    def _build_grid(self, theta, garment):
        """Build sampling grid from control point offsets using bilinear interp."""
        b = garment.size(0)
        g = self.grid_size

        # Base grid of control points in [-1, 1]
        x = torch.linspace(-1, 1, g, device=garment.device)
        y = torch.linspace(-1, 1, g, device=garment.device)
        gx, gy = torch.meshgrid(x, y, indexing='xy')
        base = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=-1)  # g^2 x 2
        base = base.unsqueeze(0).expand(b, -1, -1)                     # B x g^2 x 2

        # Add predicted offsets (clamped to avoid extreme warps)
        ctrl = base + theta.clamp(-0.5, 0.5)

        # Upsample control points to full image grid using grid_sample
        ctrl = ctrl.view(b, g, g, 2).permute(0, 3, 1, 2)  # B x 2 x g x g
        grid = F.interpolate(ctrl, size=(self.img_h, self.img_w),
                             mode='bilinear', align_corners=True)
        grid = grid.permute(0, 2, 3, 1)  # B x H x W x 2
        return grid

    def forward(self, person_with_pose, garment):
        f_p = self.person_extractor(person_with_pose)   # B x C x h x w
        f_g = self.garment_extractor(garment)           # B x C x h x w
        corr = self.correlation(f_p, f_g)               # B x hw x h x w
        theta = self.theta_reg(corr)                    # B x g^2 x 2
        grid  = self._build_grid(theta, garment)        # B x H x W x 2
        warped_garment = F.grid_sample(garment, grid,
                                        mode='bilinear',
                                        padding_mode='border',
                                        align_corners=True)
        return warped_garment, grid

print('GMM architecture defined')

In [ ]:
# ── Try-On Module (TOM) — Composites warped garment onto person ──────────────

class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=not normalize)]
        if normalize: layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2))
        if dropout: layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_ch), nn.ReLU()
        ]
        if dropout: layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip): return self.block(torch.cat([x, skip], 1))


class TryOnModule(nn.Module):
    """
    U-Net based Try-On Module.
    Input:  [person(3) + warped_garment(3) + pose_heatmap(18)] = 24 channels
    Output: [rendered_person(3) + alpha_mask(1)] = 4 channels
    """
    def __init__(self, in_ch=24, base_ch=64):
        super().__init__()
        # Encoder
        self.d1 = UNetDown(in_ch,       base_ch,    normalize=False)
        self.d2 = UNetDown(base_ch,     base_ch*2)
        self.d3 = UNetDown(base_ch*2,   base_ch*4)
        self.d4 = UNetDown(base_ch*4,   base_ch*8,  dropout=0.5)
        self.d5 = UNetDown(base_ch*8,   base_ch*8,  dropout=0.5)
        self.d6 = UNetDown(base_ch*8,   base_ch*8,  dropout=0.5)
        self.d7 = UNetDown(base_ch*8,   base_ch*8,  normalize=False, dropout=0.5)
        # Decoder
        self.u1 = UNetUp(base_ch*8,     base_ch*8,  dropout=0.5)
        self.u2 = UNetUp(base_ch*8*2,   base_ch*8,  dropout=0.5)
        self.u3 = UNetUp(base_ch*8*2,   base_ch*8,  dropout=0.5)
        self.u4 = UNetUp(base_ch*8*2,   base_ch*4)
        self.u5 = UNetUp(base_ch*4*2,   base_ch*2)
        self.u6 = UNetUp(base_ch*2*2,   base_ch)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(base_ch*2, 4, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)
        d5 = self.d5(d4)
        d6 = self.d6(d5)
        d7 = self.d7(d6)
        u1 = self.u1(d7, d6)
        u2 = self.u2(u1, d5)
        u3 = self.u3(u2, d4)
        u4 = self.u4(u3, d3)
        u5 = self.u5(u4, d2)
        u6 = self.u6(u5, d1)
        out = self.final(torch.cat([u6, x[:, :64]], 1) if x.size(1) >= 64
                         else self.final(torch.cat([u6, u6], 1)))
        rendered = out[:, :3]                        # RGB output
        alpha    = (out[:, 3:4] + 1) / 2            # [0, 1] mask
        # Composite: alpha * rendered + (1-alpha) * person
        person   = x[:, :3]
        composed = alpha * rendered + (1 - alpha) * person
        return composed, alpha


print('TryOnModule (U-Net) defined')

# Quick dimension check
gmm = GMM(grid_size=5, img_h=256, img_w=192).to(DEVICE)
tom = TryOnModule(in_ch=24, base_ch=64).to(DEVICE)

dummy_person = torch.randn(2, 21, 256, 192).to(DEVICE)  # 3 + 18
dummy_garment= torch.randn(2, 3,  256, 192).to(DEVICE)
warped, grid = gmm(dummy_person, dummy_garment)
print(f'GMM output — warped: {warped.shape}, grid: {grid.shape}')

tom_input = torch.randn(2, 24, 256, 192).to(DEVICE)
composed, alpha = tom(tom_input)
print(f'TOM output — composed: {composed.shape}, alpha: {alpha.shape}')

## Step 6: Dataset Loader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


def make_pose_heatmap(keypoints: dict, h: int, w: int, sigma: float = 8.0) -> np.ndarray:
    """
    Generate 18-channel Gaussian heatmap from pose keypoints.
    One channel per key landmark used in draping.
    """
    DRAPING_KPS = [
        'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
        'left_wrist',    'right_wrist',    'left_hip',   'right_hip',
        'left_knee',     'right_knee',     'left_ankle', 'right_ankle',
        'nose',          'left_ear',       'right_ear',
        'left_eye',      'right_eye',      'mouth_left',
    ]
    heatmaps = np.zeros((18, h, w), dtype=np.float32)
    ys, xs   = np.mgrid[0:h, 0:w].astype(np.float32)

    for i, kp_name in enumerate(DRAPING_KPS):
        if kp_name not in keypoints:
            continue
        kp  = keypoints[kp_name]
        vis = kp.get('visibility', 1.0)
        if vis < 0.2:
            continue
        cx, cy = kp['x'], kp['y']
        heatmaps[i] = np.exp(-((xs - cx)**2 + (ys - cy)**2) / (2 * sigma**2))

    return heatmaps   # 18 x H x W


class SareeDataset(Dataset):
    def __init__(self, pairs: list, pose_index: dict, img_h=256, img_w=192,
                 augment=True):
        self.pairs      = pairs
        self.pose_index = pose_index
        self.img_h      = img_h
        self.img_w      = img_w

        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

        if augment:
            self.aug = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, p=0.3),
                A.GaussNoise(var_limit=(5, 20), p=0.2),
            ])
        else:
            self.aug = None

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        person_path  = pair['person']
        garment_path = pair['garment']

        # Load images
        person_img  = np.array(Image.open(person_path).convert('RGB').resize(
                          (self.img_w, self.img_h), Image.LANCZOS))
        garment_img = np.array(Image.open(garment_path).convert('RGB').resize(
                          (self.img_w, self.img_h), Image.LANCZOS))

        # Augmentation (applied consistently to both)
        if self.aug:
            aug_p = self.aug(image=person_img)
            person_img  = aug_p['image']
            aug_g = self.aug(image=garment_img)
            garment_img = aug_g['image']

        # Load pose heatmap
        pose_path = self.pose_index.get(person_path)
        if pose_path and os.path.exists(pose_path):
            with open(pose_path) as f:
                pose_data = json.load(f)
            kp = pose_data['keypoints']
            # Scale keypoints to current image size
            orig_w = pose_data['image_size']['w']
            orig_h = pose_data['image_size']['h']
            for name in kp:
                kp[name]['x'] = kp[name]['x'] * self.img_w / orig_w
                kp[name]['y'] = kp[name]['y'] * self.img_h / orig_h
            heatmap = make_pose_heatmap(kp, self.img_h, self.img_w)
        else:
            heatmap = np.zeros((18, self.img_h, self.img_w), dtype=np.float32)

        # Convert to tensors
        person_t  = self.transform(person_img)    # 3 x H x W
        garment_t = self.transform(garment_img)   # 3 x H x W
        heatmap_t = torch.from_numpy(heatmap)     # 18 x H x W

        # Person + pose for GMM input
        person_pose = torch.cat([person_t, heatmap_t], dim=0)  # 21 x H x W

        return {
            'person':       person_t,       # 3 x H x W
            'garment':      garment_t,      # 3 x H x W
            'person_pose':  person_pose,    # 21 x H x W
            'pose_heatmap': heatmap_t,      # 18 x H x W
            'target':       person_t,       # GT = person image (self-supervised)
        }

print('SareeDataset defined')

## Step 7: Training Loop

In [ ]:
# Load pairs and pose index
with open('/kaggle/working/datasets/clean/pairs.json') as f:
    all_pairs = json.load(f)
with open('/kaggle/working/datasets/clean/pose_index.json') as f:
    pose_index = json.load(f)

# Filter to only pairs with pose data
pairs_with_pose = [p for p in all_pairs if p['person'] in pose_index]
print(f'Pairs with pose: {len(pairs_with_pose)} / {len(all_pairs)}')

# Train/Val split
import random
random.seed(42)
random.shuffle(pairs_with_pose)
n_val   = max(1, int(len(pairs_with_pose) * 0.1))
val_p   = pairs_with_pose[:n_val]
train_p = pairs_with_pose[n_val:]
print(f'Train: {len(train_p)}  |  Val: {len(val_p)}')

# Datasets & loaders
train_ds = SareeDataset(train_p, pose_index, augment=True)
val_ds   = SareeDataset(val_p,   pose_index, augment=False)

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
# ── Loss Functions ────────────────────────────────────────────────────────────

class VGGPerceptualLoss(nn.Module):
    """Perceptual loss using VGG16 features — better for cloth texture."""
    def __init__(self):
        super().__init__()
        import torchvision.models as models
        vgg = models.vgg16(pretrained=False)
        # Load pretrained weights safely
        try:
            vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        except Exception:
            pass
        self.features = nn.Sequential(*list(vgg.features)[:16]).eval()
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, pred, target):
        pred_f   = self.features(pred)
        target_f = self.features(target)
        return F.l1_loss(pred_f, target_f)


def grid_regularization_loss(grid):
    """Penalise large deformations in warp grid (keeps warping smooth)."""
    dx = grid[:, :, 1:, :] - grid[:, :, :-1, :]
    dy = grid[:, 1:, :, :] - grid[:, :-1, :, :]
    return (dx.pow(2).mean() + dy.pow(2).mean()) * 0.5


# Initialise models and losses
gmm = GMM(grid_size=5, img_h=256, img_w=192).to(DEVICE)
tom = TryOnModule(in_ch=24, base_ch=64).to(DEVICE)
vgg_loss = VGGPerceptualLoss().to(DEVICE)

# Optimisers
opt_gmm = torch.optim.Adam(gmm.parameters(), lr=1e-4, betas=(0.5, 0.999))
opt_tom = torch.optim.Adam(tom.parameters(), lr=1e-4, betas=(0.5, 0.999))

# LR schedulers
sch_gmm = torch.optim.lr_scheduler.StepLR(opt_gmm, step_size=10, gamma=0.5)
sch_tom = torch.optim.lr_scheduler.StepLR(opt_tom, step_size=10, gamma=0.5)

print('Models, losses, optimisers ready')
total_params_gmm = sum(p.numel() for p in gmm.parameters()) / 1e6
total_params_tom = sum(p.numel() for p in tom.parameters()) / 1e6
print(f'GMM params: {total_params_gmm:.1f}M  |  TOM params: {total_params_tom:.1f}M')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────

NUM_EPOCHS  = 30
SAVE_DIR    = Path('/kaggle/working/checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

history = {'gmm_loss': [], 'tom_loss': [], 'val_loss': []}


def train_epoch(epoch):
    gmm.train(); tom.train()
    gmm_losses, tom_losses = [], []

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]')
    for batch in pbar:
        person_pose = batch['person_pose'].to(DEVICE)   # B x 21 x H x W
        garment     = batch['garment'].to(DEVICE)       # B x 3 x H x W
        person      = batch['person'].to(DEVICE)        # B x 3 x H x W
        heatmap     = batch['pose_heatmap'].to(DEVICE)  # B x 18 x H x W

        # ── GMM forward ──────────────────────────────────────────────────────
        opt_gmm.zero_grad()
        warped_garment, grid = gmm(person_pose, garment)

        # GMM losses: L1 reconstruction + grid regularisation
        l1_gmm   = F.l1_loss(warped_garment, person)  # Warped garment should match person
        reg_gmm  = grid_regularization_loss(grid)
        loss_gmm = l1_gmm + 0.01 * reg_gmm
        loss_gmm.backward()
        torch.nn.utils.clip_grad_norm_(gmm.parameters(), 1.0)
        opt_gmm.step()
        gmm_losses.append(loss_gmm.item())

        # ── TOM forward ──────────────────────────────────────────────────────
        opt_tom.zero_grad()
        warped_garment_d = warped_garment.detach()
        tom_input        = torch.cat([person, warped_garment_d, heatmap], dim=1)  # 3+3+18=24
        composed, alpha  = tom(tom_input)

        # TOM losses: L1 + perceptual
        l1_tom   = F.l1_loss(composed, person)
        perc_tom = vgg_loss(
            (composed + 1) / 2,          # de-normalise to [0,1] for VGG
            (person   + 1) / 2
        )
        loss_tom = l1_tom + 0.2 * perc_tom
        loss_tom.backward()
        torch.nn.utils.clip_grad_norm_(tom.parameters(), 1.0)
        opt_tom.step()
        tom_losses.append(loss_tom.item())

        pbar.set_postfix({'GMM': f'{loss_gmm.item():.4f}',
                          'TOM': f'{loss_tom.item():.4f}'})

    return np.mean(gmm_losses), np.mean(tom_losses)


@torch.no_grad()
def validate_epoch(epoch):
    gmm.eval(); tom.eval()
    val_losses = []

    for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Val]',
                      leave=False):
        person_pose = batch['person_pose'].to(DEVICE)
        garment     = batch['garment'].to(DEVICE)
        person      = batch['person'].to(DEVICE)
        heatmap     = batch['pose_heatmap'].to(DEVICE)

        warped_garment, _ = gmm(person_pose, garment)
        tom_input         = torch.cat([person, warped_garment, heatmap], dim=1)
        composed, _       = tom(tom_input)

        val_losses.append(F.l1_loss(composed, person).item())

    return np.mean(val_losses)


# ── Run Training ──────────────────────────────────────────────────────────────
best_val  = float('inf')
for epoch in range(NUM_EPOCHS):
    gmm_l, tom_l = train_epoch(epoch)
    val_l        = validate_epoch(epoch)
    sch_gmm.step()
    sch_tom.step()

    history['gmm_loss'].append(gmm_l)
    history['tom_loss'].append(tom_l)
    history['val_loss'].append(val_l)

    print(f'Epoch {epoch+1:3d} | GMM: {gmm_l:.4f} | TOM: {tom_l:.4f} | Val: {val_l:.4f}')

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'gmm_state': gmm.state_dict(),
            'tom_state': tom.state_dict(),
            'opt_gmm':   opt_gmm.state_dict(),
            'opt_tom':   opt_tom.state_dict(),
            'history':   history,
        }, SAVE_DIR / f'checkpoint_ep{epoch+1:03d}.pth')
        print(f'  ✓ Checkpoint saved: epoch {epoch+1}')

    # Save best model
    if val_l < best_val:
        best_val = val_l
        torch.save({'gmm_state': gmm.state_dict(),
                    'tom_state': tom.state_dict()},
                   SAVE_DIR / 'best_model.pth')
        print(f'  ✓ Best model saved (val: {best_val:.4f})')

print('Training complete!')

## Step 8: Plot Training Curves & Sample Outputs

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['gmm_loss']) + 1)

axes[0].plot(epochs, history['gmm_loss'], 'b-o', label='GMM Loss', markersize=4)
axes[0].plot(epochs, history['val_loss'],  'r-o', label='Val Loss',  markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('GMM Training Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history['tom_loss'], 'g-o', label='TOM Loss', markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('TOM Training Loss'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=100)
plt.show()

# Visualise sample outputs
gmm.eval(); tom.eval()
sample_batch = next(iter(val_loader))

with torch.no_grad():
    pp = sample_batch['person_pose'].to(DEVICE)
    g  = sample_batch['garment'].to(DEVICE)
    p  = sample_batch['person'].to(DEVICE)
    h  = sample_batch['pose_heatmap'].to(DEVICE)
    wg, _ = gmm(pp, g)
    comp, alpha = tom(torch.cat([p, wg, h], 1))

def to_img(t):
    return ((t[0].cpu().permute(1,2,0).numpy() + 1) / 2 * 255).clip(0,255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
titles = ['Person', 'Garment', 'Warped Garment', 'Composed', 'Alpha Mask']
images = [to_img(p), to_img(g), to_img(wg), to_img(comp),
          (alpha[0,0].cpu().numpy() * 255).astype(np.uint8)]
cmaps  = [None, None, None, None, 'gray']
for ax, img, title, cmap in zip(axes, images, titles, cmaps):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title); ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_outputs.png', dpi=100)
plt.show()

## Step 9: Export Model Weights for Flask Integration

In [ ]:
# Export final model weights — download these and put in backend/models/

EXPORT_DIR = Path('/kaggle/working/saree_tryon_weights')
EXPORT_DIR.mkdir(exist_ok=True)

# Load best checkpoint
best_ckpt = torch.load(SAVE_DIR / 'best_model.pth', map_location='cpu')

# Save GMM weights
torch.save(best_ckpt['gmm_state'], EXPORT_DIR / 'gmm.pth')
print('Saved: gmm.pth')

# Save TOM weights
torch.save(best_ckpt['tom_state'], EXPORT_DIR / 'tom.pth')
print('Saved: tom.pth')

# Save model config
config = {
    'gmm': {'grid_size': 5, 'img_h': 256, 'img_w': 192},
    'tom': {'in_ch': 24, 'base_ch': 64},
    'img_h': 256, 'img_w': 192,
    'normalize_mean': [0.5, 0.5, 0.5],
    'normalize_std':  [0.5, 0.5, 0.5],
}
with open(EXPORT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved: config.json')

# Also save training stats
with open(EXPORT_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

# Zip for easy download
shutil.make_archive('/kaggle/working/saree_tryon_weights', 'zip', EXPORT_DIR)
size_mb = os.path.getsize('/kaggle/working/saree_tryon_weights.zip') / 1e6
print(f'\n✅ Export complete!')
print(f'   Download: /kaggle/working/saree_tryon_weights.zip ({size_mb:.1f} MB)')
print(f'   Contents: gmm.pth, tom.pth, config.json')
print(f'   Place these in: backend/models/')